# AI-Powered Agentic Customer Support System — Milestone 2

**Builds on Milestone 1.** This notebook re-establishes the knowledge base, vector
store, and the three tools from Milestone 1, then adds:

1. A **Calculator Tool** for arithmetic (totals, tax, discounts, etc.).
2. A **Human Escalation Tool** for unresolved issues.
3. A **system prompt** designed for reasoning and autonomous tool selection.
4. A **ReAct agent** built with `langgraph.prebuilt.create_react_agent()`, wired to
   all five tools, with conversation memory for multi-turn support.
5. Sample queries demonstrating the agent choosing the right tool(s) on its own.

> Set `OPENAI_API_KEY` in a `.env` file before running — the LLM/embedding cells need real network access to the OpenAI API.


## 1. Environment Setup

In [1]:
import os
import re
import glob
import ast
import operator
import uuid
import pandas as pd
import docx
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_chroma import Chroma
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    print("WARNING: OPENAI_API_KEY is not set. Set it before running the LLM/agent cells below.")
else:
    print("OpenAI API key detected.")

LLM_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
DATA_DIR = "data"
KB_DIR = os.path.join(DATA_DIR, "knowledge_base")
CHROMA_DIR = "chroma_db"


OpenAI API key detected.


## 2. Recreate the Milestone 1 Foundation

Load the structured data, rebuild the knowledge-base vector store, and redefine the
three Milestone 1 tools (`knowledge_base_search`, `order_lookup`, `sales_analytics`)
so this notebook is self-contained.


In [2]:
orders_df = pd.read_csv(os.path.join(DATA_DIR, "orders.csv"))
sales_df = pd.read_csv(os.path.join(DATA_DIR, "sales_data.csv"), parse_dates=["Date"])
print(f"Orders: {orders_df.shape}, Sales records: {sales_df.shape}")


Orders: (100, 10), Sales records: (900, 7)


In [3]:
def load_docx_text(path: str) -> str:
    d = docx.Document(path)
    return "\n".join(p.text for p in d.paragraphs if p.text.strip())


def load_knowledge_base(kb_dir: str) -> list[Document]:
    docs = []
    for path in glob.glob(os.path.join(kb_dir, "*.docx")):
        docs.append(Document(page_content=load_docx_text(path), metadata={"source": os.path.basename(path)}))
    return docs


raw_documents = load_knowledge_base(KB_DIR)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=75, separators=["\n\n", "\n", ". ", " ", ""])
chunked_documents = text_splitter.split_documents(raw_documents)

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vector_store = Chroma.from_documents(
    documents=chunked_documents,
    embedding=embeddings,
    collection_name="support_knowledge_base",
    persist_directory=CHROMA_DIR,
)
print(f"Vector store ready with {vector_store._collection.count()} chunks.")


Vector store ready with 6 chunks.


In [4]:
@tool
def knowledge_base_search(query: str) -> str:
    """Search product manuals, FAQs, and customer support policies for an answer
    to the customer's question. Use this for questions about warranty, returns,
    refunds, payment methods, delivery times, or product usage/troubleshooting."""
    results = vector_store.similarity_search(query, k=3)
    if not results:
        return "No relevant information found in the knowledge base."
    return "\n\n".join(f"Source: {r.metadata['source']}\n{r.page_content}" for r in results)


@tool
def order_lookup(order_id: str = None, email: str = None) -> str:
    """Look up order details by order_id (primary, unique identifier) or by
    customer email (returns all orders for that customer). Provide at least
    one of order_id or email."""
    if not order_id and not email:
        return "Please provide an order_id or an email address to look up an order."
    if order_id:
        try:
            oid = int(str(order_id).strip())
        except ValueError:
            return f"'{order_id}' is not a valid order ID."
        match = orders_df[orders_df["Order_ID"] == oid]
        if match.empty:
            return f"No order found with Order_ID {oid}."
        return match.to_dict(orient="records")[0].__repr__()
    match = orders_df[orders_df["Email"].str.lower() == email.strip().lower()]
    if match.empty:
        return f"No orders found for email {email}."
    return "\n".join(str(r) for r in match.to_dict(orient="records"))


@tool
def sales_analytics(
    metric: str = "revenue",
    group_by: str = "Product",
    top_n: int = 5,
    start_date: str = None,
    end_date: str = None,
) -> str:
    """Analyze sales_data.csv. metric: 'revenue', 'profit', or 'units_sold'.
    group_by: 'Product', 'Category', or 'Region'. top_n: how many rows to return,
    ranked highest first. start_date / end_date: optional 'YYYY-MM-DD' filters."""
    metric_map = {"revenue": "Revenue", "profit": "Profit", "units_sold": "Units_Sold"}
    if metric not in metric_map:
        return f"Unknown metric '{metric}'. Choose from: {list(metric_map)}."
    if group_by not in ("Product", "Category", "Region"):
        return f"Unknown group_by '{group_by}'. Choose from: Product, Category, Region."
    df = sales_df.copy()
    if start_date:
        df = df[df["Date"] >= pd.to_datetime(start_date)]
    if end_date:
        df = df[df["Date"] <= pd.to_datetime(end_date)]
    if df.empty:
        return "No sales records found in that date range."
    col = metric_map[metric]
    summary = df.groupby(group_by)[col].sum().sort_values(ascending=False).head(top_n)
    lines = [f"Top {len(summary)} {group_by.lower()}(s) by {metric}:"]
    lines += [f"  {name}: {value:,.0f}" for name, value in summary.items()]
    return "\n".join(lines)


print("Milestone 1 tools redefined: knowledge_base_search, order_lookup, sales_analytics")


Milestone 1 tools redefined: knowledge_base_search, order_lookup, sales_analytics


## 3. New Tool — Calculator

Handles arithmetic customers or the agent itself might need (order totals, tax,
discounts, unit conversions). Implemented with a restricted AST evaluator instead of
`eval()` so it can only ever perform arithmetic — no code execution risk.


In [5]:
_ALLOWED_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.Mod: operator.mod,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}


def _safe_eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("Unsupported or unsafe expression.")


@tool
def calculator(expression: str) -> str:
    """Evaluate a arithmetic expression, e.g. totals, taxes, or discounts.
    Supports +, -, *, /, %, ** and parentheses. Example: '(3 * 36000) * 1.18'
    for 3 units at 36000 plus 18% tax."""
    try:
        cleaned = expression.strip()
        tree = ast.parse(cleaned, mode="eval")
        result = _safe_eval(tree.body)
        return f"{cleaned} = {result:,.2f}"
    except Exception as e:
        return f"Could not evaluate '{expression}': {e}"


# Quick tests
print(calculator.invoke({"expression": "(3 * 36000) * 1.18"}))
print(calculator.invoke({"expression": "72000 / 2 - 5000"}))
print(calculator.invoke({"expression": "import os"}))  # should safely reject


(3 * 36000) * 1.18 = 127,440.00
72000 / 2 - 5000 = 31,000.00
Could not evaluate 'import os': invalid syntax (<unknown>, line 1)


## 4. New Tool — Human Escalation

Used when the agent cannot resolve an issue (out of scope, customer is frustrated,
policy exception needed, etc.). Creates a ticket record so a human agent can follow
up, and returns a clear confirmation message to the customer.


In [6]:
ESCALATIONS_LOG = "escalations.csv"


@tool
def human_escalation(issue_summary: str, order_id: str = None, email: str = None, urgency: str = "normal") -> str:
    """Escalate an issue to a human support agent when it cannot be resolved with
    the available tools (e.g. damaged product needing a replacement decision,
    a policy exception, or a frustrated customer). urgency: 'low', 'normal', or 'high'."""
    ticket_id = f"ESC-{uuid.uuid4().hex[:8].upper()}"
    record = {
        "Ticket_ID": ticket_id,
        "Order_ID": order_id or "",
        "Email": email or "",
        "Urgency": urgency,
        "Issue_Summary": issue_summary,
        "Status": "Open",
    }
    row_df = pd.DataFrame([record])
    header = not os.path.exists(ESCALATIONS_LOG)
    row_df.to_csv(ESCALATIONS_LOG, mode="a", index=False, header=header)
    return (
        f"Ticket {ticket_id} created (urgency: {urgency}). "
        f"A human support agent will follow up{f' regarding order {order_id}' if order_id else ''} shortly."
    )


# Quick test
print(human_escalation.invoke({
    "issue_summary": "Customer received a cracked laptop screen and wants a replacement, not a repair.",
    "order_id": "1004",
    "urgency": "high",
}))
print(pd.read_csv(ESCALATIONS_LOG))


Ticket ESC-8C983979 created (urgency: high). A human support agent will follow up regarding order 1004 shortly.
      Ticket_ID  Order_ID  Email Urgency  \
0  ESC-8C983979      1004    NaN    high   

                                       Issue_Summary Status  
0  Customer received a cracked laptop screen and ...   Open  


## 5. System Prompt Design

The prompt below is what makes tool *selection* autonomous rather than hardcoded: it
tells the agent what each tool is for, when to prefer one over another, and when to
stop and escalate instead of guessing. The individual tool docstrings (above) are what
the agent actually sees as tool descriptions at call time — this system prompt sets the
overall reasoning policy on top of that.


In [7]:
SYSTEM_PROMPT = """You are a helpful, professional customer support agent for an electronics retailer.

You have access to five tools:
- knowledge_base_search: for questions about product usage, warranty, returns, refunds, payment methods, or policies.
- order_lookup: for questions about a specific order or a customer's order history. Use order_id when the customer gives one; use email when they don't have an order ID handy.
- sales_analytics: for internal/analytical questions about revenue, profit, or units sold (e.g. "what are our top-selling products"). Do not use this to answer questions about a specific customer's order.
- calculator: for any arithmetic — totals, tax, discounts, unit conversions. Never do math yourself; always call this tool so the number is verified.
- human_escalation: for issues you cannot resolve with the tools above, for policy exceptions, or when the customer is frustrated and asks for a human. Always confirm the ticket number to the customer.

Reasoning policy:
1. Think step by step about what the customer actually needs before acting.
2. Choose the single most relevant tool for each step. Use multiple tools in sequence if the request needs it (e.g. look up an order, then calculate a refund amount).
3. If a tool returns no result or an error, try a reasonable follow-up (e.g. rephrase a search) before giving up.
4. If you cannot resolve the issue after reasonable attempts, or the customer explicitly asks for a human, use human_escalation rather than guessing or making a promise you cannot verify.
5. Never fabricate order details, policy terms, or numbers. Only state facts returned by a tool.
6. Keep responses concise, friendly, and specific to what the customer asked.
"""
print(SYSTEM_PROMPT)


You are a helpful, professional customer support agent for an electronics retailer.

You have access to five tools:
- knowledge_base_search: for questions about product usage, warranty, returns, refunds, payment methods, or policies.
- order_lookup: for questions about a specific order or a customer's order history. Use order_id when the customer gives one; use email when they don't have an order ID handy.
- sales_analytics: for internal/analytical questions about revenue, profit, or units sold (e.g. "what are our top-selling products"). Do not use this to answer questions about a specific customer's order.
- calculator: for any arithmetic — totals, tax, discounts, unit conversions. Never do math yourself; always call this tool so the number is verified.
- human_escalation: for issues you cannot resolve with the tools above, for policy exceptions, or when the customer is frustrated and asks for a human. Always confirm the ticket number to the customer.

Reasoning policy:
1. Think step 

## 6. Build the ReAct Agent

`create_react_agent()` wires the LLM, the tool list, and the system prompt into a
LangGraph agent. An `InMemorySaver` checkpointer is attached so the agent keeps
conversation state across turns (multi-turn support) as long as calls share the same
`thread_id`.


In [8]:
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

tools = [knowledge_base_search, order_lookup, sales_analytics, calculator, human_escalation]

checkpointer = InMemorySaver()

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

print("ReAct agent built with tools:", [t.name for t in tools])


ReAct agent built with tools: ['knowledge_base_search', 'order_lookup', 'sales_analytics', 'calculator', 'human_escalation']


C:\Users\91949\AppData\Local\Temp\ipykernel_16660\3620585794.py:7: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [9]:
def ask_agent(message: str, thread_id: str = "demo-thread"):
    """Send a message to the agent and print the reasoning trace + final answer."""
    config = {"configurable": {"thread_id": thread_id}}
    result = agent.invoke({"messages": [("user", message)]}, config=config)
    print(f"USER: {message}\n")
    for m in result["messages"]:
        role = m.__class__.__name__
        if role == "AIMessage" and getattr(m, "tool_calls", None):
            for tc in m.tool_calls:
                print(f"  [tool call] {tc['name']}({tc['args']})")
        elif role == "ToolMessage":
            print(f"  [tool result] {m.content[:200]}")
        elif role == "AIMessage" and m.content:
            print(f"\nAGENT: {m.content}")
    print("\n" + "=" * 80 + "\n")
    return result


## 7. Demonstrate Autonomous Tool Selection

Each query below is deliberately answerable by a *different* tool (or combination of
tools), with no hint about which one to use — the agent has to decide on its own from
the system prompt and tool descriptions.


In [10]:
# 1. Knowledge base search
ask_agent("What is the warranty period on the Dell Inspiron laptop, and how do I claim it?")


USER: What is the warranty period on the Dell Inspiron laptop, and how do I claim it?

  [tool call] knowledge_base_search({'query': 'Dell Inspiron laptop warranty period'})
  [tool call] knowledge_base_search({'query': 'how to claim warranty for Dell Inspiron laptop'})
  [tool result] Source: Customer_Support_Policies.docx
Customer Support Policies
Warranty
Laptops/Mobiles: 1 year. Accessories: 6 months.
Returns
30-day return window for eligible products.
Refunds
Refunds processed 
  [tool result] Source: Laptop_Manual.docx
Dell Inspiron 15 User Guide
Power button starts the laptop. Battery life up to 8 hours. Reset BIOS by pressing F2 during startup. Wi-Fi troubleshooting: restart router and r

AGENT: The warranty period for the Dell Inspiron laptop is 1 year. 

To claim the warranty, you typically need to follow these steps:
1. **Contact Customer Support**: Reach out to Dell's customer support through their official website or phone number.
2. **Provide Information**: Have your lapt

{'messages': [HumanMessage(content='What is the warranty period on the Dell Inspiron laptop, and how do I claim it?', additional_kwargs={}, response_metadata={}, id='441898e5-a71f-4f88-8fb5-918b470cc32b'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 822, 'total_tokens': 883, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ee57c47580', 'id': 'chatcmpl-E4TRsxr8RA2vir7nBElvhdHXfE1LR', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f8a82-3a0d-73b3-8feb-c5351d4c65ce-0', tool_calls=[{'name': 'knowledge_base_search', 'args': {'query': 'Dell Inspiron laptop warranty period'}, 'id': 

In [11]:
# 2. Order lookup by order_id
ask_agent("Can you check the status of order 1001?")


USER: Can you check the status of order 1001?

  [tool call] knowledge_base_search({'query': 'Dell Inspiron laptop warranty period'})
  [tool call] knowledge_base_search({'query': 'how to claim warranty for Dell Inspiron laptop'})
  [tool result] Source: Customer_Support_Policies.docx
Customer Support Policies
Warranty
Laptops/Mobiles: 1 year. Accessories: 6 months.
Returns
30-day return window for eligible products.
Refunds
Refunds processed 
  [tool result] Source: Laptop_Manual.docx
Dell Inspiron 15 User Guide
Power button starts the laptop. Battery life up to 8 hours. Reset BIOS by pressing F2 during startup. Wi-Fi troubleshooting: restart router and r

AGENT: The warranty period for the Dell Inspiron laptop is 1 year. 

To claim the warranty, you typically need to follow these steps:
1. **Contact Customer Support**: Reach out to Dell's customer support through their official website or phone number.
2. **Provide Information**: Have your laptop's serial number and proof of purchase

{'messages': [HumanMessage(content='What is the warranty period on the Dell Inspiron laptop, and how do I claim it?', additional_kwargs={}, response_metadata={}, id='441898e5-a71f-4f88-8fb5-918b470cc32b'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 822, 'total_tokens': 883, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ee57c47580', 'id': 'chatcmpl-E4TRsxr8RA2vir7nBElvhdHXfE1LR', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f8a82-3a0d-73b3-8feb-c5351d4c65ce-0', tool_calls=[{'name': 'knowledge_base_search', 'args': {'query': 'Dell Inspiron laptop warranty period'}, 'id': 

In [12]:
# 3. Order lookup by email
ask_agent("Can you pull up all orders placed by user1002@mail.com?")


USER: Can you pull up all orders placed by user1002@mail.com?

  [tool call] knowledge_base_search({'query': 'Dell Inspiron laptop warranty period'})
  [tool call] knowledge_base_search({'query': 'how to claim warranty for Dell Inspiron laptop'})
  [tool result] Source: Customer_Support_Policies.docx
Customer Support Policies
Warranty
Laptops/Mobiles: 1 year. Accessories: 6 months.
Returns
30-day return window for eligible products.
Refunds
Refunds processed 
  [tool result] Source: Laptop_Manual.docx
Dell Inspiron 15 User Guide
Power button starts the laptop. Battery life up to 8 hours. Reset BIOS by pressing F2 during startup. Wi-Fi troubleshooting: restart router and r

AGENT: The warranty period for the Dell Inspiron laptop is 1 year. 

To claim the warranty, you typically need to follow these steps:
1. **Contact Customer Support**: Reach out to Dell's customer support through their official website or phone number.
2. **Provide Information**: Have your laptop's serial number and p

{'messages': [HumanMessage(content='What is the warranty period on the Dell Inspiron laptop, and how do I claim it?', additional_kwargs={}, response_metadata={}, id='441898e5-a71f-4f88-8fb5-918b470cc32b'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 822, 'total_tokens': 883, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ee57c47580', 'id': 'chatcmpl-E4TRsxr8RA2vir7nBElvhdHXfE1LR', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f8a82-3a0d-73b3-8feb-c5351d4c65ce-0', tool_calls=[{'name': 'knowledge_base_search', 'args': {'query': 'Dell Inspiron laptop warranty period'}, 'id': 

In [13]:
# 4. Sales analytics
ask_agent("What were our top 3 products by revenue in Q1 2025 (Jan 1 to Mar 31)?")


USER: What were our top 3 products by revenue in Q1 2025 (Jan 1 to Mar 31)?

  [tool call] knowledge_base_search({'query': 'Dell Inspiron laptop warranty period'})
  [tool call] knowledge_base_search({'query': 'how to claim warranty for Dell Inspiron laptop'})
  [tool result] Source: Customer_Support_Policies.docx
Customer Support Policies
Warranty
Laptops/Mobiles: 1 year. Accessories: 6 months.
Returns
30-day return window for eligible products.
Refunds
Refunds processed 
  [tool result] Source: Laptop_Manual.docx
Dell Inspiron 15 User Guide
Power button starts the laptop. Battery life up to 8 hours. Reset BIOS by pressing F2 during startup. Wi-Fi troubleshooting: restart router and r

AGENT: The warranty period for the Dell Inspiron laptop is 1 year. 

To claim the warranty, you typically need to follow these steps:
1. **Contact Customer Support**: Reach out to Dell's customer support through their official website or phone number.
2. **Provide Information**: Have your laptop's seria

{'messages': [HumanMessage(content='What is the warranty period on the Dell Inspiron laptop, and how do I claim it?', additional_kwargs={}, response_metadata={}, id='441898e5-a71f-4f88-8fb5-918b470cc32b'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 822, 'total_tokens': 883, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ee57c47580', 'id': 'chatcmpl-E4TRsxr8RA2vir7nBElvhdHXfE1LR', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f8a82-3a0d-73b3-8feb-c5351d4c65ce-0', tool_calls=[{'name': 'knowledge_base_search', 'args': {'query': 'Dell Inspiron laptop warranty period'}, 'id': 

In [14]:
# 5. Calculator
ask_agent("If I buy 3 units of the Samsung Galaxy S24 at 36000 each, what's the total with 18% tax?")


USER: If I buy 3 units of the Samsung Galaxy S24 at 36000 each, what's the total with 18% tax?

  [tool call] knowledge_base_search({'query': 'Dell Inspiron laptop warranty period'})
  [tool call] knowledge_base_search({'query': 'how to claim warranty for Dell Inspiron laptop'})
  [tool result] Source: Customer_Support_Policies.docx
Customer Support Policies
Warranty
Laptops/Mobiles: 1 year. Accessories: 6 months.
Returns
30-day return window for eligible products.
Refunds
Refunds processed 
  [tool result] Source: Laptop_Manual.docx
Dell Inspiron 15 User Guide
Power button starts the laptop. Battery life up to 8 hours. Reset BIOS by pressing F2 during startup. Wi-Fi troubleshooting: restart router and r

AGENT: The warranty period for the Dell Inspiron laptop is 1 year. 

To claim the warranty, you typically need to follow these steps:
1. **Contact Customer Support**: Reach out to Dell's customer support through their official website or phone number.
2. **Provide Information**: Have 

{'messages': [HumanMessage(content='What is the warranty period on the Dell Inspiron laptop, and how do I claim it?', additional_kwargs={}, response_metadata={}, id='441898e5-a71f-4f88-8fb5-918b470cc32b'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 822, 'total_tokens': 883, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ee57c47580', 'id': 'chatcmpl-E4TRsxr8RA2vir7nBElvhdHXfE1LR', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f8a82-3a0d-73b3-8feb-c5351d4c65ce-0', tool_calls=[{'name': 'knowledge_base_search', 'args': {'query': 'Dell Inspiron laptop warranty period'}, 'id': 

In [15]:
# 6. Human escalation
ask_agent(
    "My laptop from order 1004 arrived with a cracked screen and I want a replacement, "
    "not a repair -- I'd like to speak to a human about this."
)


USER: My laptop from order 1004 arrived with a cracked screen and I want a replacement, not a repair -- I'd like to speak to a human about this.

  [tool call] knowledge_base_search({'query': 'Dell Inspiron laptop warranty period'})
  [tool call] knowledge_base_search({'query': 'how to claim warranty for Dell Inspiron laptop'})
  [tool result] Source: Customer_Support_Policies.docx
Customer Support Policies
Warranty
Laptops/Mobiles: 1 year. Accessories: 6 months.
Returns
30-day return window for eligible products.
Refunds
Refunds processed 
  [tool result] Source: Laptop_Manual.docx
Dell Inspiron 15 User Guide
Power button starts the laptop. Battery life up to 8 hours. Reset BIOS by pressing F2 during startup. Wi-Fi troubleshooting: restart router and r

AGENT: The warranty period for the Dell Inspiron laptop is 1 year. 

To claim the warranty, you typically need to follow these steps:
1. **Contact Customer Support**: Reach out to Dell's customer support through their official website 

{'messages': [HumanMessage(content='What is the warranty period on the Dell Inspiron laptop, and how do I claim it?', additional_kwargs={}, response_metadata={}, id='441898e5-a71f-4f88-8fb5-918b470cc32b'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 822, 'total_tokens': 883, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ee57c47580', 'id': 'chatcmpl-E4TRsxr8RA2vir7nBElvhdHXfE1LR', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f8a82-3a0d-73b3-8feb-c5351d4c65ce-0', tool_calls=[{'name': 'knowledge_base_search', 'args': {'query': 'Dell Inspiron laptop warranty period'}, 'id': 

In [16]:
# 7. Multi-turn conversation (same thread_id) -- tests conversation memory
ask_agent("What is your return policy?", thread_id="multi-turn-demo")
ask_agent("And does that apply to the order I have under user1002@mail.com?", thread_id="multi-turn-demo")


USER: What is your return policy?

  [tool call] knowledge_base_search({'query': 'return policy'})
  [tool result] Source: Customer_Support_Policies.docx
Customer Support Policies
Warranty
Laptops/Mobiles: 1 year. Accessories: 6 months.
Returns
30-day return window for eligible products.
Refunds
Refunds processed 

AGENT: Our return policy allows for a 30-day return window for eligible products. Refunds are processed within 7 business days after the return is received. If you have any specific questions about a return, feel free to ask!


USER: And does that apply to the order I have under user1002@mail.com?

  [tool call] knowledge_base_search({'query': 'return policy'})
  [tool result] Source: Customer_Support_Policies.docx
Customer Support Policies
Warranty
Laptops/Mobiles: 1 year. Accessories: 6 months.
Returns
30-day return window for eligible products.
Refunds
Refunds processed 

AGENT: Our return policy allows for a 30-day return window for eligible products. Refunds are process

{'messages': [HumanMessage(content='What is your return policy?', additional_kwargs={}, response_metadata={}, id='be143fd1-04eb-4220-8492-266523ce00e1'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 809, 'total_tokens': 826, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ee57c47580', 'id': 'chatcmpl-E4TSTHhiHFvRGP5VqrGVHq4XEjQ00', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f8a82-cf11-75f0-8776-a4283163459e-0', tool_calls=[{'name': 'knowledge_base_search', 'args': {'query': 'return policy'}, 'id': 'call_FQ0I4dx5YLpVQpgCIUXRWa3c', 'type': 'tool_call'}], invalid_tool_calls=

## 8. Milestone 2 Summary

- Added **`calculator`** (safe AST-based arithmetic evaluator) and **`human_escalation`**
  (creates a ticket in `escalations.csv`) as LangChain tools.
- Designed a **system prompt** that governs reasoning and tool-selection policy across
  all five tools, including when to escalate instead of guessing.
- Built a **ReAct agent** with `langgraph.prebuilt.create_react_agent()`, wired to all
  five tools and an `InMemorySaver` checkpointer for multi-turn memory.
- Demonstrated the agent autonomously selecting the correct tool (or sequence of tools)
  across seven sample queries, including a multi-turn exchange.

**Next (Milestone 3):** multi-agent workflows (e.g. a supervisor/router agent
delegating to specialist sub-agents) and transparent, structured reasoning traces
surfaced to the end user.
